In [ ]:
%pip install -q polars plotly gdown python-dotenv kailash-kaizen


# ════════════════════════════════════════════════════════════════════════
# ASCENT09 — Exercise 3: RAG Fundamentals
# ════════════════════════════════════════════════════════════════════════
# OBJECTIVE: Build a RAG pipeline from scratch — document chunking,
#   embedding generation, vector similarity search — over Singapore
#   regulatory documents.
#
# TASKS:
#   1. Chunk documents with overlap strategy
#   2. Generate embeddings via Delegate
#   3. Build simple vector store (cosine similarity)
#   4. Implement retrieval with top-k
#   5. Generate answers with retrieved context via Delegate
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import math
import os

import polars as pl

from kaizen_agents import Delegate

from shared import ASCENTDataLoader
from shared.kailash_helpers import setup_environment

setup_environment()

if not os.environ.get("OPENAI_API_KEY"):
    print("\u26a0 OPENAI_API_KEY not set \u2014 skipping LLM exercises.")
    print("  Set it in .env to run this exercise with real LLM calls.")
    import sys

    sys.exit(0)

model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
print(f"LLM Model: {model}")



### Data Loading


In [ ]:
loader = ASCENTDataLoader()
regulations = loader.load("ascent09", "sg_regulations.parquet")

print(f"Loaded {regulations.height:,} regulation sections")
print(f"Columns: {regulations.columns}")



## TASK 1: Chunk documents with overlap strategy


In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> list[str]:
    """Split text into overlapping chunks.
    If text fits in one chunk, return it directly. Otherwise slide a window of
    chunk_size characters, stepping by (chunk_size - overlap) each iteration.
    Prefer breaking at sentence boundaries (last '.' or '\\n' in the window).
    Return only non-empty stripped chunks.
    """
    # TODO: Implement the sliding-window chunker with sentence-boundary preference.
    # Return [text] if len(text) <= chunk_size. Otherwise build chunks list,
    # advancing start by (end - overlap) each iteration.
    ____
    ____


# Chunk all regulation documents
all_chunks = []
texts = regulations.select("text").to_series().to_list()
sections = (
    regulations.select("section").to_series().to_list()
    if "section" in regulations.columns
    else ["unknown"] * len(texts)
)

for i, (text, section) in enumerate(zip(texts, sections)):
    doc_chunks = chunk_text(text, chunk_size=500, overlap=100)
    for j, chunk in enumerate(doc_chunks):
        all_chunks.append(
            {"doc_idx": i, "chunk_idx": j, "section": section, "text": chunk}
        )

chunks_df = pl.DataFrame(all_chunks)
print(f"\n=== Document Chunking ===")
print(f"Total documents: {len(texts)}")
print(f"Total chunks: {chunks_df.height}")
print(f"Avg chunks per doc: {chunks_df.height / max(len(texts), 1):.1f}")
print(f"Sample chunk: {all_chunks[0]['text'][:200]}...")



## TASK 2: Generate embeddings via Delegate


In [ ]:
async def generate_embedding(text: str, delegate: Delegate) -> list[float]:
    """Generate an 8-dimensional pseudo-embedding by prompting the Delegate.
    The 8 dimensions represent: [topic_finance, topic_legal, topic_tech,
    topic_compliance, sentiment, formality, specificity, complexity].
    Ask the model to return exactly 8 comma-separated floats in [-1, 1].
    Parse the response; pad with 0.0 if fewer than 8 values are returned.
    """
    # TODO: Build the embedding prompt for the text[:300], stream the Delegate
    # response, parse comma-separated floats, pad to length 8, return the list.
    ____
    ____


async def embed_chunks(chunk_texts: list[str]) -> list[list[float]]:
    """Generate embeddings for a list of chunks using a shared Delegate."""
    # TODO: Create Delegate(model=model, budget_usd=2.0), iterate chunk_texts,
    # call generate_embedding for each, collect into a list and return.
    ____
    ____


chunk_subset = [c["text"] for c in all_chunks[:20]]
embeddings  = await embed_chunks(chunk_subset)
print(f"\n=== Embeddings ===")
print(f"Generated {len(embeddings)} embeddings")
print(f"Embedding dimension: {len(embeddings[0])}")
print(f"Sample embedding: {embeddings[0]}")



## TASK 3: Build simple vector store (cosine similarity)


In [ ]:
def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    """Compute cosine similarity between two vectors."""
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


class SimpleVectorStore:
    """Minimal vector store using cosine similarity for retrieval."""

    def __init__(self):
        self.documents: list[str] = []
        self.embeddings: list[list[float]] = []
        self.metadata: list[dict] = []

    def add(self, text: str, embedding: list[float], meta: dict | None = None):
        """Add a document with its embedding to the store."""
        # TODO: Append text, embedding, and meta (defaulting to {}) to the
        # three parallel lists.
        ____

    def search(self, query_embedding: list[float], top_k: int = 3) -> list[dict]:
        """Find the top-k most similar documents by cosine similarity.
        Score every stored embedding, sort descending, return top_k dicts
        with keys 'text', 'score', 'metadata'.
        """
        # TODO: Compute cosine_similarity(query_embedding, emb) for each stored
        # embedding, sort by score descending, return top_k result dicts.
        ____
        ____


store = SimpleVectorStore()
for i, (text, emb) in enumerate(zip(chunk_subset, embeddings)):
    store.add(text, emb, {"chunk_idx": i, "section": all_chunks[i].get("section", "")})

print(f"\n=== Vector Store ===")
print(f"Documents indexed: {len(store.documents)}")



## TASK 4: Implement retrieval with top-k


In [ ]:
async def retrieve(query: str, top_k: int = 3) -> list[dict]:
    """Embed a query and retrieve the most relevant chunks from the store."""
    # TODO: Create Delegate(model=model, budget_usd=0.5), call generate_embedding
    # on the query, call store.search with the result.
    ____
    ____


test_query = (
    "What are the compliance requirements for financial institutions in Singapore?"
)
retrieved  = await retrieve(test_query, top_k=3)
print(f"\n=== Retrieval (top-3) ===")
print(f"Query: {test_query}")
for i, result in enumerate(retrieved):
    print(f"\n  Result {i+1} (score: {result['score']:.3f}):")
    print(f"    {result['text'][:200]}...")



## TASK 5: Generate answers with retrieved context via Delegate


In [ ]:
async def rag_answer(query: str) -> str:
    """Full RAG pipeline: embed query → retrieve top-3 chunks → generate answer.
    Build context by joining retrieved chunk texts with '\\n\\n---\\n\\n'.
    Instruct the Delegate to answer using ONLY the provided context.
    If context is insufficient, the model should say so.
    """
    # TODO: Create Delegate (budget_usd=1.0). Embed query, retrieve top-3,
    # build context string, craft a grounded-answer prompt, stream response.
    ____
    ____


queries = [
    "What are the compliance requirements for financial institutions in Singapore?",
    "What penalties apply for regulatory violations?",
    "How should companies handle data protection under Singapore law?",
]

print(f"\n=== RAG Q&A ===")
for query in queries:
    answer  = await rag_answer(query)
    print(f"\nQ: {query}")
    print(f"A: {answer[:300]}...")

print("\n✓ Exercise 3 complete — RAG pipeline with chunking, embeddings, and retrieval")

